# **Semantic Web & Linked Open Data with Python**
## บทที่ 02 — RDF

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/semantic-web/blob/main/code/ch02/ch02_notebook.ipynb)

> ## *Author* : [Rathachai CHAWUTHAI](https://rathachai.creatier.pro/) , Ph.D
> ### *Affiliation* : Computer Engineering, King Mongkut's Institute of Technology Ladkrabang (KMITL)
> #### *Updated Date* : 2026-08-23
---

<a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a><br />This work (course text) is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Creative Commons Attribution 4.0 International License</a>. Sample code cells are additionally released under the MIT License.

**References:**
*   https://www.w3.org/TR/rdf11-concepts/
*   https://www.w3.org/TR/turtle/
*   https://rdflib.readthedocs.io/

📖 เอกสารประกอบบทนี้: [`02-rdf/`](https://github.com/Rathachai/semantic-web/tree/main/02-rdf)

---

### Setup

In [1]:
!pip install -q rdflib

In [2]:
import rdflib
from rdflib import Graph, URIRef, BNode, Literal, Namespace
from rdflib.namespace import RDF, RDFS, FOAF, XSD

print(rdflib.__version__)

7.6.0


## To Create Nodes

RDF มี term 3 ชนิด: **URIRef**, **BNode**, **Literal**

In [3]:
alice = URIRef("http://example.org/people/alice")

In [4]:
alice

rdflib.term.URIRef('http://example.org/people/alice')

In [5]:
linda = BNode()

In [6]:
linda

rdflib.term.BNode('Nb71dadef747449bebd724e4ba0c4cdcb')

In [7]:
name = Literal("Alice")

In [8]:
name

rdflib.term.Literal('Alice')

In [9]:
birth = Literal("1990-05-01", datatype=XSD.date)

In [10]:
birth

rdflib.term.Literal('1990-05-01', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#date'))

## To Create Nodes from Namespaces

In [11]:
ex = Namespace("http://example.org/people/")

In [12]:
ex.alice

rdflib.term.URIRef('http://example.org/people/alice')

In [13]:
FOAF.knows

rdflib.term.URIRef('http://xmlns.com/foaf/0.1/knows')

## To Create Triples และ Graph

In [14]:
g = Graph()
g.bind('ex', ex)
g.bind('foaf', FOAF)

In [15]:
g.add((ex.alice, RDF.type, FOAF.Person))
g.add((ex.alice, FOAF.name, Literal('Alice')))

<Graph identifier=Nf81de78cc4de460fb978b2c7adb6628b (<class 'rdflib.graph.Graph'>)>

In [16]:
g.add((ex.alice, FOAF.knows, ex.bob))
g.add((ex.alice, FOAF.knows, ex.carol))

<Graph identifier=Nf81de78cc4de460fb978b2c7adb6628b (<class 'rdflib.graph.Graph'>)>

In [17]:
print(g.serialize(format="turtle"))

@prefix ex: <http://example.org/people/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .

ex:alice a foaf:Person ;
    foaf:knows ex:bob,
        ex:carol ;
    foaf:name "Alice" .




## รูปแบบเต็ม vs รูปแบบย่อ (Turtle shorthand)

โหลด Turtle ที่เขียนแบบ**เต็ม**ก่อน แล้วเทียบกับแบบ**ย่อ**ที่ได้ความหมายเหมือนกันทุกประการ

In [18]:
full_ttl = """
@prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix ex:   <http://example.org/people/> .

ex:bob rdf:type foaf:Person .
ex:bob foaf:name "Bob" .
ex:bob foaf:knows ex:alice .
"""
g_full = Graph().parse(data=full_ttl, format="turtle")
print(f"{len(g_full)} triples")

3 triples


In [19]:
short_ttl = """
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix ex:   <http://example.org/people/> .

ex:bob a foaf:Person ;
    foaf:name "Bob" ;
    foaf:knows ex:alice .
"""
g_short = Graph().parse(data=short_ttl, format="turtle")
print(f"{len(g_short)} triples")

3 triples


In [20]:
from rdflib.compare import isomorphic
isomorphic(g_full, g_short)   # ความหมายเหมือนกันไหม?

True

## Blank Node — ใช้แทนสิ่งที่ไม่ต้องมีชื่อ

In [21]:
addr_ttl = """
@prefix ex: <http://example.org/people/> .
ex:alice ex:address [ ex:city "Bangkok" ; ex:postcode "10330" ] .
"""
g_addr = Graph().parse(data=addr_ttl, format="turtle")
for s, p, o in g_addr:
    print(s, p, o)

nf39a8a64486a4458842577e7211925dbb1 http://example.org/people/postcode 10330
http://example.org/people/alice http://example.org/people/address nf39a8a64486a4458842577e7211925dbb1
nf39a8a64486a4458842577e7211925dbb1 http://example.org/people/city Bangkok


## RDF Collection (list ที่มีลำดับ)

In [22]:
list_ttl = """
@prefix ex: <http://example.org/> .
ex:recipe ex:steps ( "Boil water" "Add pasta" "Drain" ) .
"""
g_list = Graph().parse(data=list_ttl, format="turtle")
print(f"{len(g_list)} triples สำหรับ list 3 สมาชิก (2n+1 = 7)")

7 triples สำหรับ list 3 สมาชิก (2n+1 = 7)


## Serialize เป็นรูปแบบอื่น

In [23]:
print(g.serialize(format="xml")[:400])

<?xml version="1.0" encoding="utf-8"?>
<rdf:RDF
   xmlns:foaf="http://xmlns.com/foaf/0.1/"
   xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#"
>
  <rdf:Description rdf:about="http://example.org/people/alice">
    <rdf:type rdf:resource="http://xmlns.com/foaf/0.1/Person"/>
    <foaf:name>Alice</foaf:name>
    <foaf:knows rdf:resource="http://example.org/people/bob"/>
    <foaf:knows rdf:reso


In [24]:
print(g.serialize(format="json-ld", indent=2)[:400])

[
  {
    "@id": "http://example.org/people/alice",
    "@type": [
      "http://xmlns.com/foaf/0.1/Person"
    ],
    "http://xmlns.com/foaf/0.1/knows": [
      {
        "@id": "http://example.org/people/bob"
      },
      {
        "@id": "http://example.org/people/carol"
      }
    ],
    "http://xmlns.com/foaf/0.1/name": [
      {
        "@value": "Alice"
      }
    ]
  }
]


In [25]:
print(g.serialize(format="nt"))

<http://example.org/people/alice> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://xmlns.com/foaf/0.1/Person> .
<http://example.org/people/alice> <http://xmlns.com/foaf/0.1/knows> <http://example.org/people/carol> .
<http://example.org/people/alice> <http://xmlns.com/foaf/0.1/knows> <http://example.org/people/bob> .
<http://example.org/people/alice> <http://xmlns.com/foaf/0.1/name> "Alice" .



## ค้นหา triple ในกราฟ

In [26]:
list(g.objects(ex.alice, FOAF.knows))

[rdflib.term.URIRef('http://example.org/people/bob'),
 rdflib.term.URIRef('http://example.org/people/carol')]

In [27]:
g.value(ex.alice, FOAF.name)

rdflib.term.Literal('Alice')

In [28]:
[s for s in g.subjects(RDF.type, FOAF.Person)]

[rdflib.term.URIRef('http://example.org/people/alice')]

## 🎯 สรุป

- RDF term 3 ชนิด: **URIRef, BNode, Literal**
- Turtle รูปแบบเต็มกับรูปแบบย่อให้กราฟที่ **isomorphic** กันเสมอ
- Blank node สร้างด้วย `[ ]` แทนโครงสร้างที่ไม่ต้องมีชื่อ
- Collection `( )` ใช้ **2n+1 triples** สำหรับ list ที่มี n สมาชิก

ลองแก้ `short_ttl` เพิ่ม property ใหม่ แล้วเทียบ `isomorphic()` อีกครั้ง!

---

## จบบทที่ 02 🎉

ไปต่อบทถัดไป: **บทที่ 03 — RDFS**

📚 กลับหน้าคอร์ส: [https://github.com/Rathachai/semantic-web](https://github.com/Rathachai/semantic-web)

---
https://github.com/Rathachai/semantic-web

*Semantic Web & Linked Open Data with Python*